# Create Lakebase Instance (Public Preview)

This notebook creates a **Lakebase** (Databricks managed Postgres OLTP database) instance using the Python SDK.

**Documentation:** [Create a database instance](https://docs.databricks.com/aws/en/oltp/instances/create/)

**Available Regions (Public Preview):**
- `us-east-1`, `us-east-2`, `us-west-2`
- `eu-west-1`, `eu-central-1`
- `ap-southeast-1`, `ap-southeast-2`, `ap-south-1`

**Capacity Options:**
- `CU_1` - 1 Compute Unit (smallest)
- `CU_2` - 2 Compute Units (default)
- `CU_4`, `CU_8`, `CU_16` - Larger sizes

**Use Cases:**
- Data serving at low latency
- Store application state
- Feature serving for ML models


In [ ]:
# Install/upgrade Databricks SDK (Lakebase API requires latest version)
%pip install --upgrade databricks-sdk -q

# Restart Python to pick up the new SDK version
dbutils.library.restartPython()


In [ ]:
# Configuration (after Python restart)
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
LAKEBASE_INSTANCE_NAME = "lakemeter-db"
LAKEBASE_CAPACITY = "CU_1"  # Options: CU_1, CU_2, CU_4, CU_8, CU_16
RETENTION_WINDOW_DAYS = 7   # Point-in-time recovery window (2-35 days)

print(f"✅ Config: {LAKEBASE_INSTANCE_NAME} ({LAKEBASE_CAPACITY})")


In [ ]:
# Create Lakebase database instance using Python SDK
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance

w = WorkspaceClient()

# Check if instance already exists
existing_instances = list(w.database.list_database_instances())
instance_exists = any(inst.name == LAKEBASE_INSTANCE_NAME for inst in existing_instances)

if instance_exists:
    print(f"⚠️ Instance '{LAKEBASE_INSTANCE_NAME}' already exists")
    instance = next(inst for inst in existing_instances if inst.name == LAKEBASE_INSTANCE_NAME)
else:
    # Create new instance
    instance = w.database.create_database_instance(
        DatabaseInstance(
            name=LAKEBASE_INSTANCE_NAME,
            capacity=LAKEBASE_CAPACITY,
            retention_window_in_days=RETENTION_WINDOW_DAYS
        )
    )
    print(f"✅ Created Lakebase instance: {instance.name}")

# Show instance info (use getattr to handle SDK variations)
try:
    state = getattr(instance, 'state', None) or instance.as_dict().get('state', 'UNKNOWN')
    capacity = getattr(instance, 'capacity', None) or instance.as_dict().get('capacity', 'UNKNOWN')
    dns = getattr(instance, 'read_write_dns', None) or instance.as_dict().get('read_write_dns', None)
    
    print(f"   State: {state}")
    print(f"   Capacity: {capacity}")
    if dns:
        print(f"   Endpoint: {dns}")
    else:
        print(f"   Endpoint: (will be available when instance is ready)")
except Exception as e:
    print(f"   Instance created - run next cell to check status")


In [ ]:
# List all database instances
instances = list(w.database.list_database_instances())
if instances:
    print(f"{'Name':<25} {'State':<20} {'Capacity':<10} {'Creator'}")
    print("-" * 80)
    for inst in instances:
        print(f"{inst.name:<25} {str(inst.state):<20} {str(inst.capacity):<10} {inst.creator}")
else:
    print("No database instances found")


In [ ]:
# Get instance details including connection info
instance_details = w.database.get_database_instance(name=LAKEBASE_INSTANCE_NAME)
d = instance_details.as_dict()

print(f"📊 Instance Details: {LAKEBASE_INSTANCE_NAME}")
print(f"="*50)
print(f"   State: {d.get('state', 'UNKNOWN')}")
print(f"   Capacity: {d.get('capacity', 'UNKNOWN')}")
print(f"   Creator: {d.get('creator', 'UNKNOWN')}")
print(f"   Created: {d.get('creation_time', 'UNKNOWN')}")

# Connection info (available when instance is AVAILABLE)
if d.get('read_write_dns'):
    print(f"\n🔗 Connection Endpoint:")
    print(f"   {d['read_write_dns']}")
else:
    print(f"\n⏳ Instance is {d.get('state')} - connection info available when AVAILABLE")


In [ ]:
# Wait for instance to become AVAILABLE
import time

print("⏳ Waiting for Lakebase instance to be ready...")
print("   (This can take 2-5 minutes)")
print()

timeout_minutes = 10
start_time = time.time()

while True:
    inst = w.database.get_database_instance(name=LAKEBASE_INSTANCE_NAME)
    d = inst.as_dict()
    state = d.get('state', 'UNKNOWN')
    
    if 'AVAILABLE' in str(state).upper() or 'RUNNING' in str(state).upper():
        print(f"\n✅ Instance is ready!")
        print(f"   State: {state}")
        if d.get('read_write_dns'):
            print(f"   Endpoint: {d['read_write_dns']}")
        break
    
    if 'FAILED' in str(state).upper() or 'ERROR' in str(state).upper():
        print(f"\n❌ Instance creation failed: {state}")
        break
    
    elapsed = time.time() - start_time
    if elapsed > timeout_minutes * 60:
        print(f"\n⚠️ Timeout after {timeout_minutes} minutes (state: {state})")
        break
    
    print(f"   State: {state} - waiting... ({int(elapsed)}s)")
    time.sleep(15)


## Next Steps

After the instance is **AVAILABLE**:

1. **Register in Unity Catalog** - Create a database catalog to access from Spark
2. **Run `02_Create_Sync_Tables.ipynb`** - Set up data sync from Unity Catalog Delta tables
3. **Connect externally** - Use psql or any Postgres client with the connection endpoint

**Manage Instance:**
```python
# Stop instance (saves cost when not in use)
w.database.update_database_instance(
    name=LAKEBASE_INSTANCE_NAME,
    database_instance=DatabaseInstance(name=LAKEBASE_INSTANCE_NAME, stopped=True),
    update_mask="*"
)

# Start instance
w.database.update_database_instance(
    name=LAKEBASE_INSTANCE_NAME,
    database_instance=DatabaseInstance(name=LAKEBASE_INSTANCE_NAME, stopped=False),
    update_mask="*"
)
```

**Documentation:** [Lakebase Documentation](https://docs.databricks.com/aws/en/oltp/instances/create/)
